# NB_LIBRO_07 · ¿Cuánto infla el resultado la fuga residual?

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 5, sección 5.4

**La pregunta.** La partición de palma del proyecto agrupa las imágenes casi idénticas con un umbral de 4 bits. El capítulo 5 mostró que hay fotografías del mismo racimo a distancias mayores, y que algunas quedaron repartidas entre entrenamiento y prueba. ¿Cambia eso el resultado del modelo?

**El experimento.** Las 70 imágenes de prueba no se tocan. Se entrena el mismo modelo (YOLO11m, misma configuración que el NB04 del proyecto) en tres condiciones y con cinco semillas:

- **A · original:** las 312 imágenes de entrenamiento del proyecto.
- **B · purga a 12 bits:** se retiran del entrenamiento y de la validación las imágenes que comparten grupo visual (misma clase, distancia dHash ≤ 12) con alguna imagen de prueba, y del entrenamiento las que lo comparten con alguna de validación.
- **C · purga a 14 bits:** lo mismo con un umbral de 14.

Como la prueba es idéntica en las tres condiciones, la diferencia de F1 macro mide el efecto de la fuga.

**Usa (solo lectura):** `PALMA_AFRICANA_YOLO/02_DATOS_PROCESADOS` y `03_SPLITS`. No modifica nada del proyecto ni guarda modelos.

**Produce:** láminas de revisión de los pares que cruzan particiones, una figura, `valores_cap05_purga.json` y las predicciones por imagen, en `AGROVISION_LIBRO`. Al final descarga un zip.

**Cómo ejecutarlo:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* (L4 recomendada; T4 también sirve) y *Ejecutar todo*. Son 15 entrenamientos de 1 a 3 minutos cada uno. Si Colab se desconecta, vuelva a ejecutar todo: las corridas terminadas se reutilizan. Suba al chat el zip que se descarga al final.

In [ ]:
%pip install -q ultralytics==8.4.172 scikit-learn

In [ ]:
import os, re, json, math, zipfile, time, shutil, gc, random
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
LOCAL = Path(os.environ.get('AGROVISION_LOCAL', '/content'))
PALMA = BASE / 'PALMA_AFRICANA_YOLO'
RAIZ_DATOS = '02_DATOS_PROCESADOS/classification_5class_v1'
DATOS = PALMA / RAIZ_DATOS
DIR_FIG = BASE / 'AGROVISION_LIBRO' / '02_FIGURAS' / 'cap05'
DIR_VAL = BASE / 'AGROVISION_LIBRO' / '03_VALORES'
DIR_EXP = DIR_VAL / 'cap05_purga'
for d in (DIR_FIG, DIR_VAL, DIR_EXP):
    d.mkdir(parents=True, exist_ok=True)

# Configuración idéntica a la del NB04 del proyecto para YOLO11m.
PESOS, EPOCAS, LADO, PACIENCIA, LOTE = 'yolo11m-cls.pt', 60, 384, 15, 8
SEMILLAS = [2026, 2027, 2028, 2029, 2030]
UMBRAL = {'A': 4, 'B': 12, 'C': 14}                 # distancia dHash máxima dentro de un grupo
NOMBRE_COND = {'A': 'A · original', 'B': 'B · purga a 12 bits', 'C': 'C · purga a 14 bits'}
CLASES_ORD = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']      # orden alfabético, el de Ultralytics

import torch, ultralytics
from ultralytics import YOLO
from sklearn.metrics import accuracy_score, f1_score
assert torch.cuda.is_available(), 'No se detectó GPU. Active una en Entorno de ejecución > Cambiar tipo de entorno.'
GPU = torch.cuda.get_device_name(0)
print('GPU:', GPU, '| PyTorch', torch.__version__, '| Ultralytics', ultralytics.__version__)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)

## 1. Grupos visuales y purga

In [ ]:
sp = pd.read_csv(PALMA / '03_SPLITS' / 'palma_split_leakage_safe_seed2026.csv')
man = pd.read_csv(PALMA / '03_SPLITS' / 'palma_processed_manifest_seed2026.csv')
PROC = dict(zip(man['sha256_source'], man['processed_relative_path']))
sp['rel'] = [Path(PROC[s]).relative_to(RAIZ_DATOS).as_posix() for s in sp['sha256']]
assert all(r.split('/')[0] == p for r, p in zip(sp['rel'], sp['split'])), 'El manifiesto y la partición no coinciden.'
faltan = [r for r in sp['rel'] if not (DATOS / r).exists()]
assert not faltan, f'Faltan {len(faltan)} imágenes procesadas, por ejemplo {faltan[:3]}'

n = len(sp)
H = np.array([int(h, 16) for h in sp['dhash_hex']], dtype=np.uint64)
x = H[:, None] ^ H[None, :]
D = np.unpackbits(x.view(np.uint8).reshape(n, n, 8), axis=2).sum(2).astype(np.int16)
clase = sp['class_inferred'].to_numpy(dtype=object)
part = sp['split'].to_numpy(dtype=object)
misma = clase[:, None] == clase[None, :]


def grupos(umbral):
    # Conjuntos disjuntos: une las imágenes de una misma clase con distancia <= umbral.
    padre = list(range(n))

    def raiz(a):
        while padre[a] != a:
            padre[a] = padre[padre[a]]; a = padre[a]
        return a
    for i, j in zip(*np.where(np.triu(misma & (D <= umbral), 1))):
        padre[raiz(int(i))] = raiz(int(j))
    return np.array([raiz(a) for a in range(n)])


def purga(umbral):
    g = grupos(umbral)
    g_prueba, g_val = set(g[part == 'test']), set(g[part == 'val'])
    quitar = ((part == 'train') & np.isin(g, list(g_prueba | g_val))) | ((part == 'val') & np.isin(g, list(g_prueba)))
    afectadas = (part == 'test') & np.isin(g, list(set(g[part != 'test'])))
    return g, quitar, afectadas

PURGA, filas = {}, []
for cond, u in UMBRAL.items():
    g, quitar, afectadas = purga(u)
    PURGA[cond] = {'grupo': g, 'quitar': quitar, 'afectadas': afectadas}
    filas.append({'condición': NOMBRE_COND[cond], 'umbral': u, 'grupos': int(len(set(g))),
                  'retiradas de entrenamiento': int((quitar & (part == 'train')).sum()),
                  'retiradas de validación': int((quitar & (part == 'val')).sum()),
                  'imágenes de prueba con pareja fuera': int(afectadas.sum())})
    sp['quitar_' + cond], sp['afectada_' + cond] = quitar, afectadas
tabla_purga = pd.DataFrame(filas)
print(tabla_purga.to_string(index=False))
assert PURGA['A']['quitar'].sum() == 0, 'Con umbral 4 no debería retirarse nada.'
sp.drop(columns=['dhash_hex']).to_csv(DIR_EXP / 'purga_por_imagen.csv', index=False)
VALORES['purga'] = filas
VALORES['retiradas_por_clase'] = {c: {k: int(v) for k, v in pd.Series(clase[PURGA[c]['quitar']]).value_counts().items()} for c in ('B', 'C')}
VALORES['prueba_afectadas'] = {c: sorted(sp.loc[PURGA[c]['afectadas'], 'filename']) for c in ('B', 'C')}
print(VALORES['retiradas_por_clase']); print(VALORES['prueba_afectadas'])

In [ ]:
# Láminas de revisión: pares de la misma clase, entre 5 y 14 bits, con una imagen en cada partición.
PART_ES = {'train': 'entrenamiento', 'val': 'validación', 'test': 'prueba'}
ii, jj = np.where(np.triu(misma & (D >= 5) & (D <= UMBRAL['C']), 1))
cruzan = [(int(D[i, j]), int(i), int(j)) for i, j in zip(ii, jj) if part[i] != part[j]]
cruzan.sort()
VALORES['pares_revision'] = [{'d': d, 'a': sp['filename'][i], 'b': sp['filename'][j], 'part_a': part[i], 'part_b': part[j]}
                             for d, i, j in cruzan]
print('Pares que cruzan particiones entre 5 y', UMBRAL['C'], 'bits:', len(cruzan))


def miniatura(i, lado=360):
    return recorte_cuadrado(leer_rgb(DATOS / sp['rel'][i], 720), lado)

POR_LAMINA, N_COLS = 9, 3
for k in range(0, len(cruzan), POR_LAMINA):
    trozo = cruzan[k:k + POR_LAMINA]
    n_filas = math.ceil(len(trozo) / N_COLS)
    fig = plt.figure(figsize=(ANCHO_IN, 1.62 * n_filas))
    gs = fig.add_gridspec(n_filas, N_COLS, wspace=0.10, hspace=0.62, left=0.01, right=0.99, top=0.95, bottom=0.08)
    for m, (d, i, j) in enumerate(trozo):
        sub = gs[m // N_COLS, m % N_COLS].subgridspec(1, 2, wspace=0.03)
        for q, idx in enumerate((i, j)):
            ax = fig.add_subplot(sub[0, q]); ax.imshow(miniatura(idx)); sin_ejes(ax)
            ax.set_xlabel(f"{Path(sp['filename'][idx]).stem}\n{PART_ES[part[idx]]}", fontsize=5.6, color=TINTA_2, labelpad=2)
            if q == 0:
                ax.set_title(f'Distancia {d}', loc='left', fontsize=7, fontweight='bold', pad=3)
    guardar(fig, DIR_FIG / f'Rev_5_pares_que_cruzan_{k // POR_LAMINA + 1}.png'); plt.show()

## 2. Entrenamiento en las tres condiciones

In [ ]:
# Copias locales de los datos: A es la partición del proyecto; B y C, la misma sin las imágenes purgadas.
RAIZ = {}
for cond in UMBRAL:
    destino = LOCAL / f'PALMA_{cond}'
    if destino.exists():
        shutil.rmtree(destino)
    shutil.copytree(DATOS if cond == 'A' else RAIZ['A'], destino)
    for r in sp.loc[PURGA[cond]['quitar'], 'rel']:
        (destino / r).unlink()
    RAIZ[cond] = destino
    cuenta = {p: sum(1 for _ in (destino / p).glob('*/*')) for p in ('train', 'val', 'test')}
    print(NOMBRE_COND[cond], cuenta)
    assert cuenta['test'] == 70
    VALORES.setdefault('tamanos', {})[cond] = cuenta
assert VALORES['tamanos']['A'] == {'train': 312, 'val': 70, 'test': 70}


def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def evaluar(modelo, raiz, particion):
    rutas = sorted(p for p in (raiz / particion).glob('*/*') if p.suffix.lower() in EXT_IMG)
    salidas = modelo.predict(source=[str(p) for p in rutas], imgsz=LADO, batch=LOTE, device=0, verbose=False)
    filas = []
    for ruta, res in zip(rutas, salidas):
        prob = res.probs.data.detach().cpu().numpy().astype(float)
        nombres = {int(k): v for k, v in res.names.items()}
        k = int(np.argmax(prob))
        filas.append({'imagen': ruta.name, 'real': ruta.parent.name, 'pred': nombres[k], 'confianza': float(prob[k])})
    return pd.DataFrame(filas)


def metricas(t):
    f1 = f1_score(t['real'], t['pred'], labels=CLASES_ORD, average=None, zero_division=0)
    return {'exactitud': float(accuracy_score(t['real'], t['pred'])),
            'f1_macro': float(f1_score(t['real'], t['pred'], labels=CLASES_ORD, average='macro', zero_division=0)),
            'f1_por_clase': {c: float(v) for c, v in zip(CLASES_ORD, f1)}}


def corrida(cond, semilla):
    nombre = f'{cond}_seed{semilla}'
    ruta_json, ruta_csv = DIR_EXP / f'{nombre}.json', DIR_EXP / f'{nombre}_prueba.csv'
    if ruta_json.exists() and ruta_csv.exists():
        print('REUTILIZADA:', nombre)
        return json.loads(ruta_json.read_text(encoding='utf-8'))
    fijar_semilla(semilla)
    t0 = time.time()
    YOLO(PESOS).train(data=str(RAIZ[cond]), epochs=EPOCAS, imgsz=LADO, batch=LOTE, patience=PACIENCIA, device=0, workers=2,
                      seed=semilla, deterministic=True, pretrained=True, optimizer='auto', amp=True, cache=True,
                      project=str(LOCAL / 'purga_runs'), name=nombre, exist_ok=True, plots=False, verbose=False)
    minutos = (time.time() - t0) / 60
    mejor = LOCAL / 'purga_runs' / nombre / 'weights' / 'best.pt'
    assert mejor.exists(), f'No se generó {mejor}'
    modelo = YOLO(str(mejor))
    val, prueba = evaluar(modelo, RAIZ[cond], 'val'), evaluar(modelo, RAIZ[cond], 'test')
    res_csv = LOCAL / 'purga_runs' / nombre / 'results.csv'
    epocas = int(len(pd.read_csv(res_csv))) if res_csv.exists() else None
    r = {'condicion': cond, 'semilla': semilla, 'minutos': round(minutos, 2), 'epocas': epocas, 'gpu': GPU,
         'n_train': VALORES['tamanos'][cond]['train'], 'n_val': int(len(val)),
         'val': metricas(val), 'prueba': metricas(prueba)}
    prueba.assign(condicion=cond, semilla=semilla).to_csv(ruta_csv, index=False)
    ruta_json.write_text(json.dumps(r, ensure_ascii=False, indent=1), encoding='utf-8')
    print(f"FINALIZADA: {nombre} | {minutos:.1f} min | val F1 {r['val']['f1_macro']:.4f} | prueba F1 {r['prueba']['f1_macro']:.4f}")
    del modelo; torch.cuda.empty_cache(); gc.collect()
    return r

CORRIDAS = []
for semilla in SEMILLAS:                     # por semilla, para que un corte deje condiciones comparables
    for cond in UMBRAL:
        CORRIDAS.append(corrida(cond, semilla))

## 3. Resultados

In [ ]:
res = pd.DataFrame([{'condicion': r['condicion'], 'semilla': r['semilla'], 'n_train': r['n_train'], 'epocas': r['epocas'],
                     'val_f1': r['val']['f1_macro'], 'prueba_f1': r['prueba']['f1_macro'], 'prueba_exactitud': r['prueba']['exactitud'],
                     **{'f1_' + c: r['prueba']['f1_por_clase'][c] for c in CLASES_ORD}} for r in CORRIDAS])
print(res.round(4).to_string(index=False))
ancho = res.pivot(index='semilla', columns='condicion', values='prueba_f1')
resumen = {}
for cond in UMBRAL:
    g = res[res['condicion'] == cond]
    resumen[cond] = {'n_train': int(g['n_train'].iloc[0]), 'f1_media': float(g['prueba_f1'].mean()), 'f1_de': float(g['prueba_f1'].std()),
                     'f1_min': float(g['prueba_f1'].min()), 'f1_max': float(g['prueba_f1'].max()),
                     'exactitud_media': float(g['prueba_exactitud'].mean()), 'val_f1_media': float(g['val_f1'].mean()),
                     'f1_media_3_semillas': float(g[g['semilla'] <= 2028]['prueba_f1'].mean()),
                     'f1_por_clase_media': {c: float(g['f1_' + c].mean()) for c in CLASES_ORD}}
    if cond != 'A':
        dif = ancho[cond] - ancho['A']
        resumen[cond]['dif_media'] = float(dif.mean()); resumen[cond]['dif_de'] = float(dif.std())
        resumen[cond]['dif_por_semilla'] = {int(s): float(v) for s, v in dif.items()}
        resumen[cond]['semillas_en_que_baja'] = int((dif < 0).sum())
VALORES['resumen'] = resumen
VALORES['f1_proyecto_NB04'] = {2026: 0.8667, 2027: 0.8103, 2028: 0.8615}
print(json.dumps({k: {q: v for q, v in r.items() if not isinstance(v, dict)} for k, r in resumen.items()}, indent=1))

# Predicciones por imagen: ¿qué cambia entre condiciones?
pred = pd.concat([pd.read_csv(DIR_EXP / f'{c}_seed{s}_prueba.csv') for c in UMBRAL for s in SEMILLAS])
pred['acierto'] = pred['real'] == pred['pred']
pred.to_csv(DIR_EXP / 'predicciones_prueba.csv', index=False)
mat = pred.pivot_table(index='imagen', columns=['condicion', 'semilla'], values='acierto').astype(bool)
detalle = {}
for cond in ('B', 'C'):
    afect = [Path(sp['rel'][i]).name for i in np.where(PURGA[cond]['afectadas'])[0]]
    cambia = {int(s): int((mat[('A', s)] != mat[(cond, s)]).sum()) for s in SEMILLAS}
    detalle[cond] = {'imagenes_afectadas': afect,
                     'aciertos_afectadas_A': {a: int(mat.loc[a, 'A'].sum()) for a in afect},
                     'aciertos_afectadas_' + cond: {a: int(mat.loc[a, cond].sum()) for a in afect},
                     'tasa_afectadas_A': float(mat.loc[afect, 'A'].to_numpy().mean()) if afect else None,
                     'tasa_afectadas_' + cond: float(mat.loc[afect, cond].to_numpy().mean()) if afect else None,
                     'tasa_resto_A': float(mat.drop(index=afect)['A'].to_numpy().mean()),
                     'tasa_resto_' + cond: float(mat.drop(index=afect)[cond].to_numpy().mean()),
                     'predicciones_que_cambian_por_semilla': cambia}
VALORES['detalle_por_imagen'] = detalle
print(json.dumps(detalle, ensure_ascii=False, indent=1))

In [ ]:
# Figura 5.10 · F1 macro en prueba por condición y semilla.
fig, ax = plt.subplots(figsize=(ANCHO_IN, 2.6))
conds = list(UMBRAL)
for s in SEMILLAS:
    ax.plot(range(len(conds)), [ancho.loc[s, c] for c in conds], color=GRIS_FONDO, lw=0.8, zorder=1)
for k, c in enumerate(conds):
    ax.scatter([k] * len(SEMILLAS), ancho[c], s=22, color=ACENTO if c == 'A' else TINTA_2, zorder=3, edgecolor='white', linewidth=0.5)
    m = resumen[c]['f1_media']
    ax.plot([k - 0.22, k + 0.22], [m, m], color=TINTA, lw=1.4, zorder=4)
    ax.text(k + 0.26, m, coma(m, 3), va='center', fontsize=7.5, fontweight='bold')
ax.set_xticks(range(len(conds)))
ax.set_xticklabels([f"{NOMBRE_COND[c]}\n{resumen[c]['n_train']} de entrenamiento" for c in conds])
ax.set_xlim(-0.5, len(conds) - 0.3); ax.set_ylabel('F1 macro en las 70 imágenes de prueba')
lo, hi = ancho.min().min(), ancho.max().max(); ax.set_ylim(lo - 0.03, hi + 0.03)
ax.yaxis.grid(True, color=REJILLA, lw=0.6); ax.set_axisbelow(True); eje_con_coma(ax.yaxis, 2); ax.tick_params(axis='x', length=0)
guardar(fig, DIR_FIG / 'Fig_5_10_purga.png'); plt.show()

In [ ]:
VALORES['configuracion'] = {'pesos': PESOS, 'epocas': EPOCAS, 'lado': LADO, 'paciencia': PACIENCIA, 'lote': LOTE, 'semillas': SEMILLAS,
                            'umbral': UMBRAL, 'gpu': GPU, 'torch': torch.__version__, 'ultralytics': ultralytics.__version__,
                            'minutos_totales': round(float(sum(r['minutos'] for r in CORRIDAS)), 1)}
ruta_json = DIR_VAL / 'valores_cap05_purga.json'
ruta_json.write_text(json.dumps(VALORES, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
res.to_csv(DIR_EXP / 'resultados_por_corrida.csv', index=False)
ruta_zip = LOCAL / 'AgroVision_cap05_purga.zip'
with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DIR_FIG.glob('Rev_5_*.png')) + [DIR_FIG / 'Fig_5_10_purga.png']:
        z.write(f, 'cap05/' + f.name)
    for f in (ruta_json, DIR_EXP / 'resultados_por_corrida.csv', DIR_EXP / 'predicciones_prueba.csv', DIR_EXP / 'purga_por_imagen.csv'):
        z.write(f, f.name)
print('Zip de', round(ruta_zip.stat().st_size / 1e6, 1), 'MB')
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))